# Hallucination Detection / Claim Verification — Baseline Experiment

This notebook implements a **baseline claim verification system** that classifies textual claims as:

| Label | Meaning |
|---|---|
| **SUPPORTS** | The claim is supported by evidence |
| **REFUTES** | The claim is contradicted by evidence |
| **NOT ENOUGH INFO** | Insufficient evidence to decide |

**Pipeline overview:**
1. Environment setup
2. Load the FEVER dataset
3. Inspect and explore the data
4. Preprocess text
5. TF-IDF vectorization
6. Train a Logistic Regression baseline
7. Evaluate with precision / recall / F1
8. Export predictions to CSV
9. Error analysis (confusion matrix, misclassifications)
10. Discussion of future extensions

## 1. Environment Setup

In [ ]:
!pip install -q pandas scikit-learn matplotlib seaborn pyarrow

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid")
print("All imports successful.")

## 2. Load the FEVER Dataset

[FEVER](https://fever.ai/) (Fact Extraction and VERification) is a large-scale benchmark for fact verification against Wikipedia.  Each example contains a **claim** and a **label** indicating whether Wikipedia evidence *supports*, *refutes*, or is *insufficient* for the claim.

We load the pre-converted **Parquet** files directly from the Hugging Face Hub (the legacy loading script is no longer supported by recent versions of the `datasets` library).

In [ ]:
PARQUET_BASE = (
    "https://huggingface.co/datasets/fever/fever"
    "/resolve/refs%2Fconvert%2Fparquet/v1.0"
)

train_df = pd.read_parquet(f"{PARQUET_BASE}/train/0000.parquet")
val_df   = pd.read_parquet(f"{PARQUET_BASE}/labelled_dev/0000.parquet")

print(f"Training rows  : {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"\nColumns: {list(train_df.columns)}")

In [ ]:
LABEL_MAP = {0: "SUPPORTS", 1: "REFUTES", 2: "NOT ENOUGH INFO"}

def normalise_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure a 'label_text' column with readable string labels."""
    if df["label"].dtype == object:
        df["label_text"] = df["label"]
    else:
        df["label_text"] = df["label"].map(LABEL_MAP)
    return df

train_df = normalise_labels(train_df)
val_df   = normalise_labels(val_df)

print("Unique labels:", train_df["label_text"].unique().tolist())

## 3. Inspect the Dataset

Let's explore column names, sample rows, and label distribution before any modeling.

In [ ]:
print("Columns:", list(train_df.columns))
print()
train_df.head()

In [ ]:
print("=== Training set label distribution ===")
print(train_df["label_text"].value_counts())
print()
print("=== Validation set label distribution ===")
print(val_df["label_text"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, df) in zip(axes, [("Train", train_df), ("Validation", val_df)]):
    counts = df["label_text"].value_counts()
    ax.bar(counts.index, counts.values, color=["#2ecc71", "#e74c3c", "#3498db"])
    ax.set_title(f"{name} Label Distribution")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts.values) * 0.01, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
print("=== Sample claims per label ===\n")
for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
    print(f"--- {label} ---")
    samples = train_df[train_df["label_text"] == label]["claim"].head(3).tolist()
    for s in samples:
        print(f"  • {s}")
    print()

## 4. Preprocess Text

We extract the **claim** text and the corresponding **label**.  Empty or null claims are dropped.

> **Note:** In this baseline we classify using the *claim text alone* (no evidence retrieval).  This is intentional — it establishes a lower-bound baseline.  A stronger system would concatenate retrieved evidence with the claim.

In [ ]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only claim + label_text; drop rows with missing claims."""
    df = df[["claim", "label_text"]].copy()
    df = df.dropna(subset=["claim"])
    df = df[df["claim"].str.strip().astype(bool)]
    df = df.reset_index(drop=True)
    return df

train_clean = preprocess(train_df)
val_clean   = preprocess(val_df)

print(f"Train after cleaning: {len(train_clean):,}")
print(f"Val   after cleaning: {len(val_clean):,}")

In [ ]:
X_train_text = train_clean["claim"].tolist()
y_train       = train_clean["label_text"].tolist()

X_val_text = val_clean["claim"].tolist()
y_val       = val_clean["label_text"].tolist()

print(f"Training texts : {len(X_train_text):,}")
print(f"Validation texts: {len(X_val_text):,}")
print(f"Label classes   : {sorted(set(y_train))}")

## 5. Convert Text to Numerical Features (TF-IDF)

We use **TF-IDF** (Term Frequency – Inverse Document Frequency) to convert each claim string into a sparse numerical vector.  We cap the vocabulary at 50 000 features and use unigrams + bigrams to capture short phrases.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
)

X_train = tfidf.fit_transform(X_train_text)
X_val   = tfidf.transform(X_val_text)

print(f"TF-IDF matrix (train): {X_train.shape}")
print(f"TF-IDF matrix (val)  : {X_val.shape}")

## 6. Train a Baseline Classifier

We train a **Logistic Regression** model — a strong, interpretable baseline for text classification.  We use the `saga` solver which scales well to large, sparse TF-IDF matrices and supports multi-class classification out of the box.

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    solver="saga",
    multi_class="multinomial",
    C=1.0,
    random_state=42,
    n_jobs=-1,
)

print("Training Logistic Regression …")
model.fit(X_train, y_train)
print("Training complete.")

## 7. Evaluate the Model

We report **accuracy**, **precision**, **recall**, and **F1-score** (macro and per-class) on the held-out validation set.

In [ ]:
y_pred = model.predict(X_val)

print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}\n")
print("Classification Report")
print("=" * 60)
print(classification_report(y_val, y_pred, digits=4))

## 8. Save Predictions to CSV

We store every validation claim alongside its true label and the model's predicted label so results can be inspected offline.

In [ ]:
results_df = pd.DataFrame({
    "claim":           X_val_text,
    "true_label":      y_val,
    "predicted_label": y_pred.tolist(),
})

results_df["correct"] = results_df["true_label"] == results_df["predicted_label"]

output_path = "baseline_predictions.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df):,} predictions → {output_path}")
results_df.head(10)

## 9. Error Analysis

### 9a. Confusion Matrix

In [ ]:
labels_order = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
cm = confusion_matrix(y_val, y_pred, labels=labels_order)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(ax=ax, cmap="Blues", values_format=",")
ax.set_title("Confusion Matrix — Baseline (Logistic Regression + TF-IDF)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### 9b. Misclassified Examples

Let's look at some examples the model got wrong to build intuition about failure modes.

In [ ]:
misclassified = results_df[~results_df["correct"]].copy()
print(f"Total misclassified: {len(misclassified):,} / {len(results_df):,} "
      f"({len(misclassified)/len(results_df)*100:.1f}%)\n")

print("=== Sample misclassifications ===\n")
for _, row in misclassified.sample(n=min(10, len(misclassified)), random_state=42).iterrows():
    print(f"  Claim : {row['claim'][:120]}")
    print(f"  True  : {row['true_label']}")
    print(f"  Pred  : {row['predicted_label']}")
    print()

### 9c. Predicted vs True Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
                           ["true_label", "predicted_label"],
                           ["True Labels", "Predicted Labels"]):
    counts = results_df[col].value_counts().reindex(labels_order)
    ax.bar(counts.index, counts.values, color=["#2ecc71", "#e74c3c", "#3498db"])
    ax.set_title(title)
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts.values) * 0.01, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

### 9d. Per-class Accuracy Breakdown

In [ ]:
print("Per-class accuracy:")
for label in labels_order:
    mask = results_df["true_label"] == label
    acc = results_df.loc[mask, "correct"].mean()
    print(f"  {label:20s}: {acc:.4f}  ({mask.sum():,} samples)")

## 10. Quick Demo — Predict on Custom Claims

Try the trained model on your own claims.

In [ ]:
def predict_claim(claim: str) -> str:
    """Return the predicted label for an arbitrary claim string."""
    vec = tfidf.transform([claim])
    return model.predict(vec)[0]

demo_claims = [
    "The Eiffel Tower is located in Berlin.",
    "Python is a programming language.",
    "Water boils at 50 degrees Celsius at sea level.",
    "Albert Einstein developed the theory of relativity.",
    "The capital of Australia is Sydney.",
]

print("=== Demo Predictions ===\n")
for claim in demo_claims:
    label = predict_claim(claim)
    print(f"  [{label:20s}]  {claim}")

## 11. Future Extensions

This TF-IDF + Logistic Regression baseline establishes a **lower-bound** for the claim verification task.  Possible improvements for the research project:

| Extension | Description |
|---|---|
| **Evidence retrieval** | Retrieve Wikipedia passages and concatenate them with the claim before classification |
| **Transformer models** | Fine-tune BERT / RoBERTa on claim–evidence pairs for much stronger representations |
| **Domain-specific data** | Apply the pipeline to SciFact (scientific claim verification) |
| **Confidence scoring** | Use `predict_proba` to output calibrated confidence scores |
| **Cross-encoder reranking** | Rerank retrieved evidence with a cross-encoder before classification |
| **Data augmentation** | Paraphrase claims or back-translate to increase training diversity |

---

**End of baseline experiment.**